In [1]:
# pipenv install pandas seaborn plotly scikit-learn matplotlib ipywidgets ipykernel nbformat mlxtend

# EDA
import pandas as pd
import numpy as np
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns

# ML
from mlxtend.frequent_patterns import apriori, association_rules

# EDA

## Loading the dataset

In [4]:
df_transactions = pd.read_csv('./datasets/transactions_dept.csv', sep=',')
df_transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 4539 entries, 0 to 4538
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   POS Txn  4539 non-null   uint64
 1   Dept     4539 non-null   str   
 2   ID       4539 non-null   int64 
 3   Sales U  4539 non-null   int64 
dtypes: int64(2), str(1), uint64(1)
memory usage: 142.0 KB


In [5]:
df_transactions.head(10)

,POS Txn,Dept,ID,Sales U
0,16120100160021008773,0261:HOSIERY,250,2
1,16120100160021008773,0634:VITAMINS & HLTH AIDS,102,1
2,16120100160021008773,0879:PET SUPPLIES,158,2
3,16120100160021008773,0973:CANDY,175,2
4,16120100160021008773,0982:SPIRITS,176,1
5,16120100160021008773,0983:WINE,177,4
6,16120100160021008773,0991:TOBACCO,179,2
7,16120100160021008774,0597:HEALTH AIDS,93,1
8,16120100160021008774,0604:PERSONAL CARE,100,5
9,16120100160021008775,0819:PRE-RECORDED A/V,135,1


In [8]:
df_transactions.rename(
    columns={
        "POS Txn": "ID_Transaction",
        "Dept": "Department",
        "ID": "ID_Department",
        "Sales U": "Quantity_Sold"
    }, inplace=True
)

df_transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 4539 entries, 0 to 4538
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   ID_Transaction  4539 non-null   uint64
 1   Department      4539 non-null   str   
 2   ID_Department   4539 non-null   int64 
 3   Quantity_Sold   4539 non-null   int64 
dtypes: int64(2), str(1), uint64(1)
memory usage: 142.0 KB


In [ ]:
# Department Quantity
df_transactions['Department'].nunique()

160

In [10]:
# ID Department Quantity
df_transactions['ID_Department'].nunique()

160

In [12]:
# Department Quantity
df_transactions['ID_Transaction'].nunique()

2064

In [13]:
# Can a department appear more than once in the same transaction?
len(df_transactions.groupby(['ID_Transaction', 'Department'])) != len(df_transactions)

False

In [ ]:
# Check for lower or equal to zero quantity sold <= 0
len(df_transactions[df_transactions['Quantity_Sold'] <= 0])

149

In [15]:
# Removing row with quantity sold <= 0
df_transactions = df_transactions[df_transactions['Quantity_Sold'] > 0]

## Exploring the dataset based on some business questions

### Which departments have the highest transaction volume and the largest number of units sold?

In [21]:
# Number of unique transactions in which each department appears
# Transaction volume
count_department_transactions = df_transactions.value_counts('Department').head(10)

fig_transaction_volume_top10 = px.bar(count_department_transactions, color=count_department_transactions.index, orientation='h')
fig_transaction_volume_top10.update_layout(showlegend=False)
fig_transaction_volume_top10.show()

In [ ]:
# Number of unique transactions in which each department appears
# Transaction volume
sum_department_units_sold_top10 = df_transactions.groupby('Department')['Quantity_Sold'].sum().sort_values(ascending=False).head(10)

fig_units_sold_top10 = px.bar(sum_department_units_sold_top10, color=sum_department_units_sold_top10.index, orientation='h')
fig_units_sold_top10.update_layout(showlegend=False)
fig_units_sold_top10.show()

In [ ]:
# Number of unique transactions in which each department appears
# Unity Sold
count_department_transactions_top10 = df_transactions.value_counts('Department').head(10)

count_department_transactions_top10 = px.bar(count_department_transactions_top10, color=count_department_transactions_top10.index, orientation='h')
count_department_transactions_top10.update_layout(showlegend=False)

count_department_transactions_top10.show()

In [27]:
# Combine the two “Top 10” charts to display them side by side

fig_question_1 = make_subplots(rows=1, cols=2, subplot_titles=("Transactions Volume", "Units Sold"))

for trace in fig_transaction_volume_top10['data']:
    fig_question_1.add_trace(trace, row=1, col=1)

for trace in fig_units_sold_top10['data']:
    fig_question_1.add_trace(trace, row=1, col=2)

fig_question_1.update_layout(height=800, width=1300, title_text='Top 10 Departments', showlegend=False)
fig_question_1.show()

### Which departments have the highest variation in units sold per transaction?

In [35]:
# Calculate the standard deviation
std_department_units_sold_top20 = df_transactions.groupby('Department')['Quantity_Sold'].std().sort_values(ascending=False).head(20)

fig_variation_units_sold_top20 = px.bar(std_department_units_sold_top20, color=std_department_units_sold_top20.index, orientation='h')
fig_variation_units_sold_top20.update_layout(showlegend=False)
fig_variation_units_sold_top20.show()

### What is the breakdown of units sold by department across the various transactions?

In [33]:
# Evaluate the position measures for the Quantity_Sold variable
px.box(df_transactions, x='Department', y='Quantity_Sold')

### Which departments have the highest average number of units sold per transaction?

In [36]:
# Calculate the mean
mean_department_units_sold_top20 = df_transactions.groupby('Department')['Quantity_Sold'].mean().sort_values(ascending=False).head(20)

fig_mean_units_sold_top20 = px.bar(mean_department_units_sold_top20, color=mean_department_units_sold_top20.index, orientation='h')
fig_mean_units_sold_top20.update_layout(showlegend=False)
fig_mean_units_sold_top20.show()

In [37]:
# Combine the two “Top 20” charts to display them side by side

fig_question_4 = make_subplots(rows=1, cols=2, subplot_titles=("Standard Deviation", "Mean"))

for trace in fig_variation_units_sold_top20['data']:
    fig_question_4.add_trace(trace, row=1, col=1)

for trace in fig_mean_units_sold_top20['data']:
    fig_question_4.add_trace(trace, row=1, col=2)

fig_question_4.update_layout(height=800, width=1300, title_text='Top 20 Departments', showlegend=False)
fig_question_4.show()